# Éditeur mobile → Kokoro → MP4

Exporter votre projet depuis le site, puis exécuter ces cellules dans Colab depuis Chrome Android. Session CPU Python 3.12. Aucun jeton API payant. Le modèle gratuit et Chromium sont téléchargés sur la machine Colab. Les ressources gratuites et la durée de session dépendent de Google.


In [17]:
import os, subprocess, sys, json
from pathlib import Path
assert sys.version_info[:2] >= (3, 11), "Sélectionner une version de Python compatible (>= 3.11)"
REF = "main"  # Moteur publié sur la branche principale.
PROJECT = Path('/content/Destiny-Explained-Studio')
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch',REF,'--single-branch','https://github.com/Alexanst54/Destiny-Explained-Studio.git',str(PROJECT)],check=True)
else:
    # En cas de modifications locales résiduelles, on les réinitialise pour pouvoir synchroniser proprement
    subprocess.run(['git','-C',str(PROJECT),'reset','--hard'],check=True)
    subprocess.run(['git','-C',str(PROJECT),'clean','-fd'],check=True)

subprocess.run(['git','-C',str(PROJECT),'fetch','origin',f'refs/heads/{REF}:refs/remotes/origin/{REF}'],check=True)
subprocess.run(['git','-C',str(PROJECT),'switch',REF],check=True)
subprocess.run(['git','-C',str(PROJECT),'merge','--ff-only',f'origin/{REF}'],check=True)
head = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip()
upstream = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse',f'origin/{REF}'],text=True).strip()
assert head == upstream, "Le checkout local diffère de main ; utiliser une nouvelle session Colab"
print(f'Moteur chargé depuis {REF}: {head}')

Moteur chargé depuis main: f6f7b9d34d54f661e1da63540803151858857f94


In [19]:
%%bash
set -euo pipefail
cd /content
NODE_VERSION=22.23.4
NODE_ARCHIVE="node-v${NODE_VERSION}-linux-x64.tar.xz"
curl -fsSLO "https://nodejs.org/dist/v${NODE_VERSION}/${NODE_ARCHIVE}"
curl -fsSLo node-SHASUMS256.txt "https://nodejs.org/dist/v${NODE_VERSION}/SHASUMS256.txt"
awk -v f="$NODE_ARCHIVE" '$2 == f' node-SHASUMS256.txt | sha256sum --check -
tar -xJf "$NODE_ARCHIVE"
apt-get update -qq
AUDIO_PACKAGE=libasound2
if apt-cache show libasound2t64 >/dev/null 2>&1; then AUDIO_PACKAGE=libasound2t64; fi
apt-get install -y ffmpeg python3-venv libnss3 libatk-bridge2.0-0 libx11-xcb1 libxcomposite1 libxdamage1 libxrandr2 libgbm1 "$AUDIO_PACKAGE" fonts-liberation

# Correction ici : Utilisation de virtualenv à la place de venv standard pour éviter le bug ensurepip sur Colab
pip install --upgrade --quiet virtualenv
virtualenv /content/destiny-tts

/content/destiny-tts/bin/pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cpu
/content/destiny-tts/bin/pip install -r /content/Destiny-Explained-Studio/remotion/tts/requirements-kokoro.txt

node-v22.23.4-linux-x64.tar.xz: OK
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
python3-venv is already the newest version (3.12.3-0ubuntu2.1).
libnss3 is already the newest version (2:3.98-1ubuntu0.3).
libatk-bridge2.0-0t64 is already the newest version (2.52.0-1build1).
libx11-xcb1 is already the newest version (2:1.8.7-1ubuntu0.1).
libxcomposite1 is already the newest version (1:0.4.5-1build3).
libxdamage1 is already the newest version (1:1.1.6-1build1).
libxrandr2 is already the newest version (2:1.5.2-2build1).
libgbm1 is already the newest version (25.2.8-0ubuntu0.24.04.4).
libasound2t64 is already the newest version (1.2.11-1ubuntu0.3).
fonts-liberation is already the newest version (1:2.1.5-3).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
created virtual environment CPython3.13.18.final.0-64-x86_64 in 1005ms
  creator CPython3Posix(dest=/content/destiny-tts, clear=Fa

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## Importer le JSON exporté
Choisir un seul fichier JSON dans les téléchargements Android. L’original reste dans `out/editor-upload.json`. La copie pour rendu écarte les champs inconnus : le rapport les liste. Les erreurs de schéma et de durée bloquent le traitement. Aucun exemple du dépôt n’est modifié.


In [20]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, "Choisir exactement un fichier JSON"
name, data = next(iter(uploaded.items()))
assert name.lower().endswith('.json') and len(data) <= 2*1024*1024, "JSON de 2 Mo maximum"
ROOT = PROJECT / 'remotion'
(ROOT / 'out').mkdir(exist_ok=True)
(ROOT / 'out/editor-upload.json').write_bytes(data)


Saving atlas-of-destiny-demo.json to atlas-of-destiny-demo.json


17743

In [21]:
%%bash
set -euo pipefail
export PATH="/content/destiny-tts/bin:/content/node-v22.23.4-linux-x64/bin:$PATH"
export HF_HUB_DISABLE_XET=1
cd /content/Destiny-Explained-Studio/remotion
npm ci
npm run typecheck
node --experimental-strip-types scripts/import-editor.mjs out/editor-upload.json --compatible-copy
python scripts/narration.py out/editor-episode.json
npm run browser:ensure
npm run render:episode -- out/editor-episode-ready.json



added 274 packages, and audited 275 packages in 16s

51 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities

> destiny-explained-studio@0.1.0 typecheck
> tsc --noEmit

Original conservé. Copie moteur : out/editor-episode.json
{
  "omittedFromRenderCopy": [],
  "warnings": [
    "Les débuts liés à des phrases sont estimés ici. Le contrôle définitif se fait avec la voix Kokoro dans Colab."
  ]
}
atlas-of-destiny-demo: 4 scenes, 720 frames, 24s
atlas-riis: 5.200s / 6.000s, 12 words
atlas-mercury: 5.075s / 6.000s, 10 words
atlas-fundament: 4.900s / 6.000s, 12 words
atlas-europa: 5.100s / 6.000s, 9 words
atlas-of-destiny-demo: 4 scenes, 720 frames, 24s
Ready: /content/Destiny-Explained-Studio/remotion/out/editor-episode-ready.json
Timeline: 24.000s; 43 aligned words

> destiny-explained-studio@0.1.0 browser:ensure
> remotion browser ensure

Getting Headless Shell - 9.5 Mb/102.5 Mb
Getting Headless Shell - 19.1 Mb/102.5 Mb
Getting Headless Shell - 28.6 Mb/

(node:25861) [MODULE_TYPELESS_PACKAGE_JSON] Warning: Module type of file:///content/Destiny-Explained-Studio/remotion/atlas/index.ts is not specified and it doesn't parse as CommonJS.
Reparsing as ES module because module syntax was detected. This incurs a performance overhead.
To eliminate this warning, add "type": "module" to /content/Destiny-Explained-Studio/remotion/package.json.
(Use `node --trace-warnings ...` to show where the warning was created)
PREVIEW: audio disabled and subtitle timing estimated.
Voiceover is pending: preview is silent.
Word alignment is pending: preview captions use estimated timing.
(node:25882) [MODULE_TYPELESS_PACKAGE_JSON] Warning: Module type of file:///content/Destiny-Explained-Studio/remotion/src/voxel/cinematic/timeline.ts is not specified and it doesn't parse as CommonJS.
Reparsing as ES module because module syntax was detected. This incurs a performance overhead.
To eliminate this warning, add "type": "module" to /content/Destiny-Explained-Studi

## Télécharger le MP4 et les rapports
Kokoro utilise af_heart à 1× par défaut. Une narration trop longue échoue explicitement ; raccourcir le texte ou allonger la scène dans l’éditeur puis réimporter. Les ancrages de phrases sont contrôlés avec les mots réellement prononcés. Enregistrer les résultats avant la fin de la session Colab.


In [22]:
ready = json.loads((ROOT / 'out/editor-episode-ready.json').read_text())
video = (ROOT / ready['render']['output']).resolve()
assert video.is_relative_to(ROOT.resolve()) and video.suffix.lower() == '.mp4'
assert video.exists(), "Rendu MP4 absent"
files.download(str(video))
files.download(str(ROOT / 'out/editor-import-report.json'))
files.download(str(ROOT / 'out/editor-episode-ready.json'))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>